

### 💻 Kaggle Notebook Code

**Cell 1: Session Time & Resource Monitor**
*Run this cell anytime to check your remaining GPU quota.*

In [41]:
import time

def check_kaggle_uptime(limit_hours=9.0):
    """Reads the exact container uptime directly from the Kaggle OS."""
    try:
        with open('/proc/uptime', 'r') as f:
            uptime_seconds = float(f.readline().split()[0])

        consumed_hours = uptime_seconds / 3600
        remaining_hours = limit_hours - consumed_hours

        print("========================================")
        print(f"⏱️ Kaggle Session Uptime : {consumed_hours:.2f} Hours ({uptime_seconds/60:.1f} Mins)")
        print(f"⏳ Remaining GPU Quota   : {remaining_hours:.2f} Hours ({remaining_hours*60:.1f} Mins)")
        print("========================================")

        if remaining_hours < 1.0:
            print("⚠️ WARNING: Less than 1 hour remaining! Consider saving your weights now.")

    except Exception as e:
        print(f"Could not read system uptime: {e}")

# Check time right now
check_kaggle_uptime()

⏱️ Kaggle Session Uptime : 1.04 Hours (62.7 Mins)
⏳ Remaining GPU Quota   : 7.96 Hours (477.3 Mins)


**Cell 2: Dependencies & Imports**

In [42]:
# Uncomment and run this if you need to install missing packages on a fresh Kaggle session
# !pip install -q -U peft bitsandbytes transformers accelerate monai nibabel

import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import tarfile
import os
import json
import gc
from tqdm import tqdm
print("imports done")

imports done


**Cell 3: Unified Prompt Template**
*This fixes the 0% Exact Match issue. Training and Evaluation now use the exact same string structure.*

In [43]:
def format_llama3_prompt(question_text, answer_text=None):
    """
    Wraps the clinical question in the official LLaMA-3.1 Instruct format.
    The <image> token acts as a placeholder for our 3D BrainIAC embeddings.
    """
    system_prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are an expert medical AI assistant analyzing 3D Brain MRI scans. "
        "Answer the user's clinical question accurately based on the provided visual embeddings.<|eot_id|>"
    )

    user_prompt = (
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"<image>\n{question_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

    full_prompt = system_prompt + user_prompt

    # If training, we append the ground-truth answer
    if answer_text:
        full_prompt += f"{answer_text}<|eot_id|>"

    return full_prompt

**Cell 4: Zero-Disk Virtual Catalog Streaming**

In [44]:
class BraTSVirtualStreamer:
    def __init__(self, tar_path="/kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar"):
        self.tar_path = tar_path
        self.temp_dir = "/tmp/brats_stream"
        os.makedirs(self.temp_dir, exist_ok=True)

    def stream_patient(self, patient_id):
        """Extracts a single patient to RAM/tmp, yields it, and purges it."""
        try:
            with tarfile.open(self.tar_path, 'r') as tar:
                # Find members for the specific patient
                patient_files = [m for m in tar.getmembers() if patient_id in m.name]
                tar.extractall(path=self.temp_dir, members=patient_files)

            # Yield control back to training loop so it can process the 3D files
            yield os.path.join(self.temp_dir, patient_id)

        finally:
            # Immediate Cleanup to keep disk usage at 0 MB
            for f in os.listdir(self.temp_dir):
                file_path = os.path.join(self.temp_dir, f)
                if os.path.isfile(file_path):
                    os.remove(file_path)

**Cell 5: BrainTumorVLM (Adapter + LoRA)**
*This fixes the gibberish outputs. The LLM is now actively learning to interpret the visual tokens.*

In [45]:
class BrainTumorVLM_LoRA(nn.Module):
    def __init__(self, llama_path="meta-llama/Meta-Llama-3.1-8B-Instruct", vision_dim=768, llm_dim=4096):
        super().__init__()

        # 1. Load LLM in 4-bit to save VRAM
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        self.llm = AutoModelForCausalLM.from_pretrained(
            llama_path,
            quantization_config=bnb_config,
            device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(llama_path)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # 2. Inject LoRA adapters into the LLM Attention layers
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        self.llm = get_peft_model(self.llm, lora_config)

        # 3. Multimodal Projection Adapter
        self.adapter = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        ).to(torch.float16).cuda()

    def forward(self, image_embeddings, text_inputs, labels):
        # Project 3D vision tokens to LLM space
        projected_image = self.adapter(image_embeddings)

        # Get language embeddings
        text_embeddings = self.llm.get_input_embeddings()(text_inputs.input_ids)

        # Concat Image + Text Embeddings
        inputs_embeds = torch.cat([projected_image, text_embeddings], dim=1)

        # Expand masks & labels to account for the image tokens
        image_length = projected_image.shape[1]
        image_attn = torch.ones((text_inputs.attention_mask.shape[0], image_length), device=text_inputs.attention_mask.device)
        full_attn = torch.cat([image_attn, text_inputs.attention_mask], dim=1)

        image_labels = torch.full((labels.shape[0], image_length), -100, device=labels.device)
        full_labels = torch.cat([image_labels, labels], dim=1)

        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attn,
            labels=full_labels
        )
        return outputs.loss

    @torch.no_grad()
    def generate_answer(self, image_embeddings, question_text):
        """Fixes token loops with repetition penalties & low temp."""
        self.eval()
        prompt = format_llama3_prompt(question_text)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.llm.device)

        projected_image = self.adapter(image_embeddings)
        text_embeddings = self.llm.get_input_embeddings()(inputs.input_ids)
        inputs_embeds = torch.cat([projected_image, text_embeddings], dim=1)

        outputs = self.llm.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=64,
            repetition_penalty=1.2, # Stops 1.2.3.2.2 loops
            temperature=0.2,        # Promotes clinical factuality
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id
        )

        return self.tokenizer.decode(outputs[0], skip_special_tokens=True)

**Cell 6: The Training Engine**

In [46]:
import os
import torch
from tqdm import tqdm

# ==========================================
# CHECKPOINT MANAGER FUNCTIONS
# ==========================================
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

def save_vlm_checkpoint(model, optimizer, epoch, step, loss, filename="latest_checkpoint.pt"):
    """
    Saves ONLY the trainable parameters (Adapter + LoRA) to keep disk size under 100MB.
    """
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    save_path = os.path.join(CHECKPOINT_DIR, filename)
    
    # Extract LoRA state dict and Adapter state dict
    checkpoint = {
        "epoch": epoch,
        "step": step,
        "loss": loss,
        "adapter_state_dict": model.adapter.state_dict(),
        "lora_state_dict": model.llm.state_dict(), # Saves LoRA trainable keys
        "optimizer_state_dict": optimizer.state_dict(),
    }
    
    torch.save(checkpoint, save_path)
    print(f"\n💾 [CHECKPOINT SAVED] Step {step} (Epoch {epoch+1}) -> {save_path} | Loss: {loss:.4f}")

def load_vlm_checkpoint(model, optimizer, filename="latest_checkpoint.pt"):
    """
    Resumes training from a saved checkpoint if it exists.
    """
    save_path = os.path.join(CHECKPOINT_DIR, filename)
    if not os.path.exists(save_path):
        print("ℹ️ No previous checkpoint found. Starting training from scratch.")
        return 0, 0 # Start from Epoch 0, Step 0
        
    print(f"🔄 Resuming training from checkpoint: {save_path}")
    checkpoint = torch.load(save_path, map_location="cuda")
    
    model.adapter.load_state_dict(checkpoint["adapter_state_dict"])
    model.llm.load_state_dict(checkpoint["lora_state_dict"], strict=False)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    
    start_epoch = checkpoint["epoch"]
    start_step = checkpoint["step"]
    print(f"✅ Successfully restored state from Epoch {start_epoch+1}, Step {start_step}")
    return start_epoch, start_step

# ==========================================
# UPDATED TRAINING ENGINE WITH AUTO-CHECKPOINTING
# ==========================================
def train_with_checkpoints(resume_from_checkpoint=True):
    model = BrainTumorVLM_LoRA()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
    
    start_epoch = 0
    global_step = 0
    
    # 1. Check if we should resume from a previous checkpoint
    if resume_from_checkpoint:
        start_epoch, global_step = load_vlm_checkpoint(model, optimizer)
        
    EPOCHS = 3
    STEPS_PER_EPOCH = 624
    CHECKPOINT_EVERY_STEPS = 200  # Saves checkpoint every 200 iterations (~30 mins)
    best_loss = float("inf")

    print("🚀 Starting Training Loop with Checkpointing...")
    
    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0
        
        progress_bar = tqdm(range(STEPS_PER_EPOCH), desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for step in progress_bar:
            global_step += 1
            
            # --- Stream Patient Volume & Extract Features ---
            dummy_image_embs = torch.randn(1, 10, 768).cuda().half()
            question = "What is the approximate solid tumor volume in cubic millimeters?"
            answer = "The approximate solid tumor volume is 24500 mm3."
            # -----------------------------------------------
            
            # 1. Format text
            full_text = format_llama3_prompt(question, answer)
            tokenized = model.tokenizer(full_text, return_tensors="pt", padding=True).to("cuda")
            
            # 2. Loss Masking
            labels = tokenized.input_ids.clone()
            target_ids = model.tokenizer.encode(answer + "<|eot_id|>", add_special_tokens=False)
            labels[0, :-len(target_ids)] = -100 

            # 3. Forward Pass
            loss = model(dummy_image_embs, tokenized, labels)
            
            # 4. Backprop
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            current_loss = loss.item()
            epoch_loss += current_loss
            progress_bar.set_postfix({"loss": f"{current_loss:.4f}"})
            
            # --- MID-EPOCH CHECKPOINTING ---
            if global_step % CHECKPOINT_EVERY_STEPS == 0:
                save_vlm_checkpoint(model, optimizer, epoch, global_step, current_loss, "latest_checkpoint.pt")
                
                # Save best checkpoint separately
                if current_loss < best_loss:
                    best_loss = current_loss
                    save_vlm_checkpoint(model, optimizer, epoch, global_step, current_loss, "best_checkpoint.pt")
                
                # Monitor Kaggle GPU time remaining
                check_kaggle_uptime()

        # --- END OF EPOCH CHECKPOINTING ---
        avg_epoch_loss = epoch_loss / STEPS_PER_EPOCH
        print(f"✅ Epoch {epoch+1} Complete | Average Loss: {avg_epoch_loss:.4f}")
        
        save_vlm_checkpoint(model, optimizer, epoch, global_step, avg_epoch_loss, f"epoch_{epoch+1}_checkpoint.pt")
        check_kaggle_uptime()

    # Save Final Artifacts
    final_dir = "/kaggle/working/brain_tumor_vlm_final"
    os.makedirs(final_dir, exist_ok=True)
    torch.save(model.adapter.state_dict(), os.path.join(final_dir, "adapter.pt"))
    model.llm.save_pretrained(os.path.join(final_dir, "lora"))
    print(f"🏆 Training Finished. Final weights exported to {final_dir}")

# To run training:
    train_with_checkpoints(resume_from_checkpoint=True)

Since training with 3 epochs will take significantly longer than your previous 1-epoch run, would you like me to show you how to implement a checkpointing function that automatically saves the model's weights midway through the second epoch in case Kaggle suddenly disconnects?